In [ ]:
import numpy as np

import matplotlib.pyplot as plt
import matplotlib.tri as tri

def ddx_c(phi, dx):
    # centered difference on cell centers, periodic
    return (np.roll(phi, -1, axis=1) - np.roll(phi, 1, axis=1)) / (2*dx)

def ddy_c(phi, dy):
    return (np.roll(phi, -1, axis=0) - np.roll(phi, 1, axis=0)) / (2*dy)

def ddy_c_wall(phi, dy):
    """
    y-derivative on cell centers with rigid walls (nonperiodic).
    Centered differences in the interior; one-sided at j=0 and j=ny-1.
    """
    d = np.empty_like(phi)
    d[1:-1, :] = (phi[2:, :] - phi[:-2, :]) / (2*dy)
    d[0,    :] = (phi[1,  :] - phi[0,   :]) / dy
    d[-1,   :] = (phi[-1, :] - phi[-2,  :]) / dy
    return d

def avg_x_to_u(phi_c):
    # average cell-centered -> u points (faces in x)
    return 0.5 * (phi_c + np.roll(phi_c, -1, axis=1))

def avg_y_to_v(phi_c):
    # average cell-centered -> v points (faces in y)
    return 0.5 * (phi_c + np.roll(phi_c, -1, axis=0))

def avg_y_to_v_wall(phi_c):
    """
    Average cell-centered field to v-locations (between j and j+1),
    without wrapping in y.

    We store v on a (ny,nx) array for simplicity.
    Interpret v[j,:] as the meridional velocity on the SOUTH face of row j.
      - Then v[0,:] is the south boundary face  -> will be set to 0
      - v[-1,:] is the south face of last row   (interior), and the north boundary
        is represented by v_north = v at face j=ny which we don't store.
    For our simplified layout, this average is mainly used to collocate things;
    we just avoid wrap and clamp at the northern edge.
    """
    out = np.empty_like(phi_c)
    out[:-1, :] = 0.5 * (phi_c[:-1, :] + phi_c[1:, :])
    out[-1,  :] = phi_c[-1, :]  # clamp at top edge
    return out

def div_Cgrid(u, v, dx, dy):
    # u, v live on faces but stored on same (ny,nx) arrays for simplicity.
    # Interpret u(i,j) as u at (i,j+1/2), v(i,j) as v at (i+1/2,j).
    dudx = (u - np.roll(u, 1, axis=1)) / dx
    dvdy = (v - np.roll(v, 1, axis=0)) / dy
    return dudx + dvdy

def div_Cgrid_walls(u, v, dx, dy):
    """
    Divergence of fluxes (hu, hv) with periodic x and rigid y walls.

    dudx uses periodic wrap in x.
    dvdy uses one-sided at boundaries (no wrap).
    """
    dudx = (u - np.roll(u, 1, axis=1)) / dx

    dvdy = np.empty_like(v)
    dvdy[1:-1, :] = (v[1:-1, :] - v[:-2, :]) / dy
    dvdy[0,    :] = (v[0,    :] - 0.0) / dy        # south wall face is 0
    dvdy[-1,   :] = (0.0 - v[-2,   :]) / dy        # north wall face is 0

    return dudx + dvdy

def laplacian(phi, dx, dy):
    return ((np.roll(phi,-1,0) - 2*phi + np.roll(phi,1,0))/dy**2 +
            (np.roll(phi,-1,1) - 2*phi + np.roll(phi,1,1))/dx**2)

def apply_rigid_y_walls(u, v, h):
    # No-normal flow at y boundaries:
    v[0, :]  = 0.0
    v[-1, :] = 0.0
    return u, v, h

def rhs(u, v, h, g, f0, H, dx, dy, nu=0.0, wall=False ):
    """
    Shallow-water equations (nonlinear):
      du/dt = - (u·∇)u + f v - g ∂h/∂x + ν∇²u
      dv/dt = - (u·∇)v - f u - g ∂h/∂y + ν∇²v
      dh/dt = - ∇·(h u_vec) + ν∇²h   (here nu on h is optional, mild smoothing)
    On a C-grid-ish layout using simple averages.
    """

    # Gradients of h at centers, then average to u/v points
    dhdx_c = ddx_c(h, dx)
    dhdy_c = ddy_c(h, dy)
    dhdx_u = avg_x_to_u(dhdx_c)
    dhdy_v = avg_y_to_v(dhdy_c)

    # Compute advective terms using centered differences on u/v arrays
    dudx = (np.roll(u, -1, axis=1) - np.roll(u, 1, axis=1)) / (2*dx)
    dudy = (np.roll(u, -1, axis=0) - np.roll(u, 1, axis=0)) / (2*dy)
    dvdx = (np.roll(v, -1, axis=1) - np.roll(v, 1, axis=1)) / (2*dx)
    dvdy = (np.roll(v, -1, axis=0) - np.roll(v, 1, axis=0)) / (2*dy)

    # Need velocity at same points for u·∇u etc.
    # Approximate v at u-points and u at v-points
    v_on_u = avg_x_to_u(v)
    u_on_v = avg_y_to_v(u)

    adv_u = u * dudx + v_on_u * dudy
    adv_v = u_on_v * dvdx + v * dvdy

    # Momentum RHS
    du = -adv_u + f0 * v_on_u - g * dhdx_u + nu * laplacian(u, dx, dy)
    dv = -adv_v - f0 * u_on_v - g * dhdy_v + nu * laplacian(v, dx, dy)

    # Continuity: dh/dt = -div( h*u, h*v )
    # Put h at u/v points via averaging
    h_on_u = avg_x_to_u(h)
    h_on_v = avg_y_to_v(h)
    hu = h_on_u * u
    hv = h_on_v * v
    dh = -div_Cgrid(hu, hv, dx, dy) + nu * laplacian(h, dx, dy)

    return du, dv, dh

def step_rk2(u, v, h, dt, *rhs_args, **rhs_kwargs):
    du1, dv1, dh1 = rhs(u, v, h, *rhs_args, **rhs_kwargs)
    u1 = u + dt * du1
    v1 = v + dt * dv1
    h1 = h + dt * dh1
    du2, dv2, dh2 = rhs(u1, v1, h1, *rhs_args, **rhs_kwargs)
    u_new = u + 0.5*dt*(du1 + du2)
    v_new = v + 0.5*dt*(dv1 + dv2)
    h_new = h + 0.5*dt*(dh1 + dh2)
    return u_new, v_new, h_new

# ----------------- Demo run -----------------
if __name__ == "__main__":
    nx, ny = 128, 128
    Lx, Ly = 1.0e6, 1.0e6
    dx, dy = Lx/nx, Ly/ny

    g  = 9.81
    H  = 1000.0
    f0 = 1.0e-4
    nu = 200.0   # mild viscosity (m^2/s) for stability during early dev

    x = (np.arange(nx) + 0.5) * dx
    y = (np.arange(ny) + 0.5) * dy
    X, Y = np.meshgrid(x, y)

    # Initial condition: a Gaussian bump in free-surface height
    h = H + 10.0 * np.exp(-(((X - 0.5*Lx)**2 + (Y - 0.5*Ly)**2) / (2*(0.08*Lx)**2)))
    u = np.zeros((ny, nx))
    v = np.zeros((ny, nx))

    # Simple CFL-ish timestep
    c = np.sqrt(g*H)
    dt = 0.2 * min(dx, dy) / c

    
    nsteps = 2000
    hx=np.zeros( (nsteps, ny, nx ) )

    for n in range(nsteps):
        u, v, h = step_rk2(u, v, h, dt, g, f0, H, dx, dy, nu=nu)

        hx[n,:,:] = h
    
        if n % 200 == 0:
            eta = h - H
            print(f"step {n:5d}  eta range: {eta.min(): .3e} .. {eta.max(): .3e}")


In [ ]:
plt.contour( hx[0,:,:] )

In [ ]:
plt.contourf( hx[:, ny//2, :] , levels=51, cmap='plasma' )



In [ ]:
nx//2